<a href="https://colab.research.google.com/github/ravindyaparami/multi-agents-on-a-lie-group/blob/main/Drake/OOPv3_Force_Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np

class BodyConfig:
    """
    Configuration for a general rigid body.
    Defined purely by mass and inertia tensor.
    """

    def __init__(self,
                 mass: float,
                 inertia_matrix: np.ndarray,
                 initial_position=None,
                 initial_orientation=None,
                 gravity=None,
                 friction=(0.9, 0.8)):

        # ---- Fundamental physical properties ----
        self.mass = float(mass)
        self.inertia_matrix = np.array(inertia_matrix, dtype=float)

        # ---- Initial position ----
        self.initial_position = (
            np.array(initial_position, dtype=float)
            if initial_position is not None
            else np.array([0.0, 0.0, 1.0])
        )

        # Orientation stored as rotation matrix (3x3)
        self.initial_orientation = (
            np.array(initial_orientation, dtype=float)
            if initial_orientation is not None
            else np.eye(3)
        )

        # ---- Optional simulation properties ----
        self.gravity = (
            np.array(gravity, dtype=float)
            if gravity is not None
            else np.array([0.0, 0.0, -9.81])
        )

        self.friction = friction

        # ---- Validate physical correctness ----
        self._validate()

    def _validate(self):
        if self.mass <= 0:
            raise ValueError("Mass must be positive.")

        if self.inertia_matrix.shape != (3, 3):
            raise ValueError("Inertia matrix must be 3x3.")

        # Must be symmetric
        if not np.allclose(self.inertia_matrix,
                           self.inertia_matrix.T):
            raise ValueError("Inertia matrix must be symmetric.")

        # Must be positive definite
        eigenvalues = np.linalg.eigvals(self.inertia_matrix)
        if not np.all(eigenvalues > 0):
            raise ValueError(
                "Inertia matrix must be positive definite."
            )

        if self.initial_orientation.shape != (3, 3):
            raise ValueError("Initial orientation must be 3x3.")





In [ ]:
'''from pydrake.all import (
    LeafSystem,
    Value,
    ExternallyAppliedSpatialForce,
    SpatialForce
)

class ExternalForceSystem(LeafSystem):
    def __init__(self, plant, body, force_model):
        super().__init__()

        self.plant = plant
        self.body = body
        self.force_model = force_model

        # Declare output port (list of spatial forces)
        self.DeclareAbstractOutputPort(
            "spatial_forces",
            lambda: Value(list()),
            self.CalcOutput
        )

    def CalcOutput(self, context, output):

        forces = []

        if self.force_model is None:
            output.set_value(forces)
            return

        # Get plant context
        root_context = context.get_root_context()
        plant_context = self.plant.GetMyContextFromRoot(root_context)

        t = context.get_time()
        q = self.plant.GetPositions(plant_context)
        v = self.plant.GetVelocities(plant_context)

        # User returns a SpatialForce
        spatial_force = self.force_model(t, q, v)

        force = ExternallyAppliedSpatialForce()
        force.body_index = self.body.index()
        force.p_BoBq_B = np.zeros(3)
        force.F_Bq_W = spatial_force

        forces.append(force)

        output.set_value(forces)'''


In [ ]:
from pydrake.all import LeafSystem, Value, ExternallyAppliedSpatialForce
import numpy as np


class ExternalForceSystem(LeafSystem):
    def __init__(self, plant, body, force_models):
        super().__init__()
        self.plant = plant
        self.body = body
        self.force_models = force_models

        # Declare an input port to receive the plant's state
        self.DeclareVectorInputPort(
            "plant_state",
            plant.num_positions() + plant.num_velocities()
        )

        self.DeclareAbstractOutputPort(
            "spatial_forces",
            lambda: Value([ExternallyAppliedSpatialForce()]),
            self.CalcOutput
        )

    def CalcOutput(self, context, output):
        forces = []

        # Read state directly from the input port
        state = self.EvalVectorInput(context, 0).value()
        nq = self.plant.num_positions()
        q = state[:nq]
        v = state[nq:]
        t = context.get_time()

        for model in self.force_models:
            f_vec = model(t, q, v)
            force = ExternallyAppliedSpatialForce()
            force.body_index = self.body.index()
            force.p_BoBq_B = np.zeros(3)
            force.F_Bq_W = SpatialForce(tau=np.zeros(3), f=f_vec)
            forces.append(force)

        output.set_value(forces)

In [ ]:
'''from pydrake.all import *
import numpy as np

class RigidBodySimulator:
    """
    Wrapper around Drake to simulate a single rigid body
    with customizable physics.
    """
    def __init__(self, config: BodyConfig, force_model = None, time_step: float = 0.001):
        self.config = config
        self.force_model = force_model
        self.time_step = time_step
        self._build_system()

    def _build_system(self):
        # 1. Diagram & plant
        self.builder = DiagramBuilder()
        self.plant, self.scene_graph = AddMultibodyPlantSceneGraph(
            self.builder,
            MultibodyPlant(time_step=self.time_step)
        )

        # 2. Add body
        self._add_body()

        # 3. Add ground
        self._add_ground()

        # 4. Finalize plant
        self.plant.Finalize()

        # 4.1 Add external force system if provided
        if self.force_model is not None:
            self.force_system = ExternalForceSystem(
                self.plant,
                self.body,
                self.force_model
            )

            self.builder.AddSystem(self.force_system)

            self.builder.Connect(
                self.force_system.get_output_port(),
                self.plant.get_applied_spatial_force_input_port()
            )


        # 5. Meshcat visualizer (optional, for debugging)
        self.meshcat = StartMeshcat()
        MeshcatVisualizer.AddToBuilder(
            self.builder,
            self.scene_graph,
            self.meshcat
        )

        # 6. Build diagram
        self.diagram = self.builder.Build()
        self.simulator = Simulator(self.diagram)

    def _add_body(self):
        cfg = self.config

        # Convert inertia matrix to UnitInertia
        I = cfg.inertia_matrix
        m = cfg.mass

        unit_inertia = UnitInertia(
            Ixx=I[0, 0]/m,
            Iyy=I[1, 1]/m,
            Izz=I[2, 2]/m,
            Ixy=I[0, 1]/m,
            Ixz=I[0, 2]/m,
            Iyz=I[1, 2]/m
        )

        spatial_inertia = SpatialInertia(
            mass=cfg.mass,
            p_PScm_E=np.zeros(3),
            G_SP_E=unit_inertia
        )

        self.body = self.plant.AddRigidBody("body", spatial_inertia)

        # Set initial pose
        X_WB = RigidTransform(
            RotationMatrix(cfg.initial_orientation),
            cfg.initial_position
        )

        self.plant.SetDefaultFloatingBaseBodyPose(
            self.body,
            X_WB
        )

        # Minimal placeholder collision geometry
        box_size = [0.2, 0.2, 0.2]
        collision_shape = Box(box_size[0], box_size[1], box_size[2])
        self.plant.RegisterCollisionGeometry(
            self.body,
            RigidTransform(),
            collision_shape,
            "body_collision",
            CoulombFriction(*cfg.friction)
        )

        # Minimal visual geometry (for Meshcat visualization)
        visual_shape = Box(box_size[0], box_size[1], box_size[2])
        self.plant.RegisterVisualGeometry(
            self.body,
            RigidTransform(),
            visual_shape,
            "body_visual",
            [0.2, 0.6, 1.0, 1.0]  # RGBA color
        )

        # Gravity
        self.plant.mutable_gravity_field().set_gravity_vector(cfg.gravity)

    def _add_ground(self):
        ground_shape = HalfSpace()
        X_WG = RigidTransform(RollPitchYaw(np.pi, 0, 0), [0, 0, 0])

        self.plant.RegisterCollisionGeometry(
            self.plant.world_body(),
            X_WG,
            ground_shape,
            "ground_collision",
            CoulombFriction(0.9, 0.8)
        )

        self.plant.RegisterVisualGeometry(
            self.plant.world_body(),
            X_WG,
            ground_shape,
            "ground_visual",
            [0.5, 0.5, 0.5, 1.0]
        )

    def simulate(self, duration: float = 5.0, realtime_rate: float = 1.0):
        self.simulator.set_target_realtime_rate(realtime_rate)
        self.simulator.Initialize()
        self.simulator.AdvanceTo(duration)

    def get_web_url(self) -> str:
        return self.meshcat.web_url()

    def get_state(self):
        context = self.simulator.get_context()
        plant_context = self.plant.GetMyContextFromRoot(context)
        q = self.plant.GetPositions(plant_context)
        v = self.plant.GetVelocities(plant_context)
        return q, v
  '''

In [ ]:
from pydrake.all import *
import numpy as np

class RigidBodySimulator:
    """
    Wrapper around Drake to simulate a single rigid body
    with customizable physics.
    """
    def __init__(self, config: BodyConfig, force_models = None, time_step: float = 0.001):
        self.config = config
        self.force_models = force_models
        self.time_step = time_step
        self._build_system()

    def _build_system(self):
        # 1. Diagram & plant
        self.builder = DiagramBuilder()
        self.plant, self.scene_graph = AddMultibodyPlantSceneGraph(
            self.builder,
            MultibodyPlant(time_step=self.time_step)
        )

        # 2. Add body
        self._add_body()

        # 3. Add ground
        self._add_ground()

        # 4. Finalize plant
        self.plant.Finalize()

        # 4.1 Add external force system if provided
        if self.force_models:
            self.force_system = ExternalForceSystem(
                self.plant,
                self.body,
                self.force_models
            )

            self.builder.AddSystem(self.force_system)

            self.builder.Connect(
                self.plant.get_state_output_port(),
                self.force_system.get_input_port(0)
            )

            self.builder.Connect(
                self.force_system.get_output_port(),
                self.plant.get_applied_spatial_force_input_port()
            )


        # 5. Meshcat visualizer (optional, for debugging)
        self.meshcat = StartMeshcat()
        MeshcatVisualizer.AddToBuilder(
            self.builder,
            self.scene_graph,
            self.meshcat
        )

        # 6. Build diagram
        self.diagram = self.builder.Build()
        self.simulator = Simulator(self.diagram)

    def _add_body(self):
        cfg = self.config

        # Convert inertia matrix to UnitInertia
        I = cfg.inertia_matrix
        m = cfg.mass

        unit_inertia = UnitInertia(
            Ixx=I[0, 0]/m,
            Iyy=I[1, 1]/m,
            Izz=I[2, 2]/m,
            Ixy=I[0, 1]/m,
            Ixz=I[0, 2]/m,
            Iyz=I[1, 2]/m
        )

        spatial_inertia = SpatialInertia(
            mass=cfg.mass,
            p_PScm_E=np.zeros(3),
            G_SP_E=unit_inertia
        )

        self.body = self.plant.AddRigidBody("body", spatial_inertia)

        # Set initial pose
        X_WB = RigidTransform(
            RotationMatrix(cfg.initial_orientation),
            cfg.initial_position
        )

        self.plant.SetDefaultFloatingBaseBodyPose(
            self.body,
            X_WB
        )

        # Minimal placeholder collision geometry
        box_size = [0.2, 0.2, 0.2]
        collision_shape = Box(box_size[0], box_size[1], box_size[2])
        self.plant.RegisterCollisionGeometry(
            self.body,
            RigidTransform(),
            collision_shape,
            "body_collision",
            CoulombFriction(*cfg.friction)
        )

        # Minimal visual geometry (for Meshcat visualization)
        visual_shape = Box(box_size[0], box_size[1], box_size[2])
        self.plant.RegisterVisualGeometry(
            self.body,
            RigidTransform(),
            visual_shape,
            "body_visual",
            [0.2, 0.6, 1.0, 1.0]  # RGBA color
        )

        # Gravity
        self.plant.mutable_gravity_field().set_gravity_vector(cfg.gravity)

    def _add_ground(self):
        ground_shape = HalfSpace()
        X_WG = RigidTransform(RollPitchYaw(np.pi, 0, 0), [0, 0, 0])

        self.plant.RegisterCollisionGeometry(
            self.plant.world_body(),
            X_WG,
            ground_shape,
            "ground_collision",
            CoulombFriction(0.9, 0.8)
        )

        self.plant.RegisterVisualGeometry(
            self.plant.world_body(),
            X_WG,
            ground_shape,
            "ground_visual",
            [0.5, 0.5, 0.5, 1.0]
        )

    def simulate(self, duration: float = 5.0, realtime_rate: float = 1.0):
        self.simulator.set_target_realtime_rate(realtime_rate)
        self.simulator.Initialize()
        self.simulator.AdvanceTo(duration)

    def get_web_url(self) -> str:
        return self.meshcat.web_url()

    def get_state(self):
        context = self.simulator.get_context()
        plant_context = self.plant.GetMyContextFromRoot(context)
        q = self.plant.GetPositions(plant_context)
        v = self.plant.GetVelocities(plant_context)
        return q, v

In [ ]:
# ---- 1. Define cube config ----
mass = 1.0
cube_size = 0.2
inertia_matrix = np.diag([(1/6)*mass*(cube_size**2 + cube_size**2)]*3)
initial_position = [0,0,1]
rpy = RollPitchYaw(0,0,np.pi/4)
initial_orientation = rpy.ToRotationMatrix().matrix()
config = BodyConfig(mass=mass, inertia_matrix=inertia_matrix,
                    initial_position=initial_position,
                    initial_orientation=initial_orientation,
                    gravity=[0,0,-9.81])

# ---- Simple constant force along +X ----
def constant_force(t, q, v):
    return np.array([2.0,0.0,0.0])

sim = RigidBodySimulator(
    config=config,
    force_models=[constant_force],
    time_step=0.001
)

In [ ]:
import numpy as np
from math import sin, cos, pi
from pydrake.all import SpatialForce

# --- 1. Define the body ---
mass = 2.0
# Diagonal inertia for a rectangular solid
inertia_matrix = np.diag([0.1, 0.2, 0.3])

# Initial position high above the ground
initial_position = [0.0, 0.0, 1.0]

body_cfg = BodyConfig(
    mass=mass,
    inertia_matrix=inertia_matrix,
    initial_position=initial_position
)

# --- 2. Define force models ---
def sinusoidal_push(t, q, v):
    """Applies a sinusoidal force along X"""
    return np.array([5.0 * sin(2*pi*t), 0.0, 0.0])

def rotating_torque(t, q, v):
    """Applies a torque along Z (as spatial force, torque-only)"""
    return np.array([0.0, 0.0, 2.0 * cos(pi*t)])

# Combine forces
force_models = [sinusoidal_push, rotating_torque]

sim = RigidBodySimulator(
    config=body_cfg,
    force_models=force_models,
    time_step=0.001)

In [ ]:
url = sim.get_web_url()
print("Open this URL in your browser to visualize the simulation:")
print(url)

In [ ]:
# Simulate for 8 seconds at real-time speed
sim.simulate(duration=8.0, realtime_rate=0.5)


q, v = sim.get_state()

print("Positions (q):", q)
print("Velocities (v):", v)